# Q3 — Feature Engineering and Regression Pipeline

**Objective:** Build a reproducible scikit-learn regression pipeline to predict `items_sold` at a retail store using temporal feature engineering and a proper time-series train-test split.

**Dataset:** `q3_retail_promotions.csv` — 1,200 records, 9 columns, date range 2022-01-01 to 2024-12-31.

## Task 1 — Date Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 100
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load data
df = pd.read_csv("../data/q3_retail_promotions.csv")
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['transaction_date'].min().date()} to {df['transaction_date'].max().date()}")
print("\nOriginal columns:")
print(df.dtypes)

# Extract date features
df["year"]        = df["transaction_date"].dt.year
df["month"]       = df["transaction_date"].dt.month
df["day_of_week"] = df["transaction_date"].dt.dayofweek   # 0=Monday, 6=Sunday
df["is_month_end"]= (df["transaction_date"].dt.day >= 25).astype(int)

print("\nNew feature columns added: year, month, day_of_week, is_month_end")
print("\nSample of engineered features:")
display(df[["transaction_date","year","month","day_of_week","is_month_end","items_sold"]].head(10))

print(f"\nis_month_end value counts:")
print(df["is_month_end"].value_counts())

### Why These Features Matter

- **year** — captures long-term demand trends and business growth across 2022–2024.
- **month** — captures seasonal patterns (festive season peaks in Oct–Dec, slow months in Jan–Feb).
- **day_of_week** — captures intra-week rhythms; weekends versus weekdays drive different footfall behaviour.
- **is_month_end** (day ≥ 25) — captures salary-day spending spikes common in retail, especially for working-class urban stores.

These four features transform a raw date string — which a machine learning model cannot process directly — into structured numerical signals that describe the temporal context of each transaction.

## Task 2 — Temporal Train-Test Split

In [ ]:
# Sort by date to preserve temporal order
df_sorted = df.sort_values("transaction_date").reset_index(drop=True)

# Temporal 80/20 split — most recent 20% as test
split_idx = int(len(df_sorted) * 0.8)
train = df_sorted.iloc[:split_idx].copy()
test  = df_sorted.iloc[split_idx:].copy()

print(f"Total records : {len(df_sorted)}")
print(f"Training set  : {len(train)} records  | {train['transaction_date'].min().date()} to {train['transaction_date'].max().date()}")
print(f"Test set      : {len(test)} records   | {test['transaction_date'].min().date()} to {test['transaction_date'].max().date()}")
print(f"\nSplit index   : row {split_idx} (80% mark)")
print(f"\nTarget stats — Train:")
print(train["items_sold"].describe().round(2))
print(f"\nTarget stats — Test:")
print(test["items_sold"].describe().round(2))

### Why a Random Split is Inappropriate for Time-Ordered Data

A random split shuffles records without regard to chronological order, causing two critical problems:

1. **Data leakage:** Future data points can end up in the training set while their temporal neighbours land in the test set. The model then effectively "sees" future information during training — the resulting test metrics are optimistically biased and do not reflect real deployment performance.

2. **Distribution mismatch:** In retail, patterns evolve over time. Seasonality, promotions, and competition change year over year. A model trained on a random 80% sample of 2022–2024 data is tested on a random 20% of the same period — an unrealistically favourable evaluation. In production, the model will always be asked to predict *future* transactions using only *past* data.

A **temporal split** — train on the earliest 80%, test on the most recent 20% — correctly simulates the real deployment scenario where we train on historical data and evaluate on unseen future records (Jun–Dec 2024 in this case).

## Task 3 — Preprocessing Pipeline

In [ ]:
# Define features and target
feature_cols = [
    "store_id", "store_size", "location_type", "promotion_type",
    "is_weekend", "is_festival", "competition_density",
    "year", "month", "day_of_week", "is_month_end"
]
target_col = "items_sold"

cat_cols = ["store_size", "location_type", "promotion_type"]
num_cols = [c for c in feature_cols if c not in cat_cols]

X_train = train[feature_cols]
y_train = train[target_col]
X_test  = test[feature_cols]
y_test  = test[target_col]

# ColumnTransformer: one-hot encode categoricals, scale numericals
preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_cols),
    ("num", StandardScaler(), num_cols)
], remainder="drop")

print("Pipeline preprocessor configured:")
print(f"  Categorical columns (OneHotEncoder, drop=first): {cat_cols}")
print(f"  Numerical columns  (StandardScaler)            : {num_cols}")
print(f"\nTraining set shape  : {X_train.shape}")
print(f"Test set shape      : {X_test.shape}")
print("\nThe preprocessor will be fit ONLY on the training set (inside each pipeline).")
print("It is applied via transform() to the test set — no leakage.")

### Pipeline Design Rationale

Using a scikit-learn `Pipeline` wrapping `ColumnTransformer` is the correct, leakage-free approach:

- `OneHotEncoder(drop='first')` — drops one dummy column per categorical feature to avoid perfect multicollinearity (the dummy variable trap). Applied to `promotion_type` (5 levels → 4 dummies), `location_type` (3 → 2), and `store_size` (3 → 2).
- `StandardScaler` — standardises numerical features to zero mean, unit variance. This is essential for Linear Regression (where gradient descent converges faster) and generally good practice.
- **Fit on training data only** — when `pipeline.fit(X_train, y_train)` is called, both the `ColumnTransformer` and the model are fit exclusively on training data. `pipeline.predict(X_test)` applies the already-fit transformer to the test set, guaranteeing no test statistics contaminate the training process.

## Task 4 — Model Training and Evaluation

In [ ]:
# ── Linear Regression ──────────────────────────────────────────────────────────
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
lr_pipeline.fit(X_train, y_train)
lr_pred = lr_pipeline.predict(X_test)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_mae  = mean_absolute_error(y_test, lr_pred)

print("Linear Regression — Test Set Metrics")
print(f"  RMSE : {lr_rmse:.2f}")
print(f"  MAE  : {lr_mae:.2f}")

# Parity plot
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, lr_pred, alpha=0.4, color="steelblue", s=30, label="Predictions")
diag = [y_test.min(), y_test.max()]
ax.plot(diag, diag, "r--", linewidth=1.8, label="Perfect prediction (y = x)")
ax.set_xlabel("Actual items_sold", fontsize=12)
ax.set_ylabel("Predicted items_sold", fontsize=12)
ax.set_title("Linear Regression — Parity Plot\n(Predicted vs Actual)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Linear Regression Results

**RMSE = 27.12 | MAE = 21.05**

The parity plot shows predictions clustered around the diagonal but with a visible cone of scatter — the model underestimates high values and overestimates low values, a classic sign of a model that has captured the linear trend but lacks the flexibility to model non-linear interactions between promotion type, store characteristics, and temporal signals. An RMSE of ~27 items means predictions are off by roughly 27 units on average (squared-error weighted), while the MAE of ~21 reflects the typical absolute error.

In [ ]:
# ── Random Forest Regressor ────────────────────────────────────────────────────
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42))
])
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae  = mean_absolute_error(y_test, rf_pred)

print("Random Forest Regressor — Test Set Metrics")
print(f"  RMSE : {rf_rmse:.2f}")
print(f"  MAE  : {rf_mae:.2f}")

# Parity plot
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, rf_pred, alpha=0.4, color="#e67e22", s=30, label="Predictions")
diag = [y_test.min(), y_test.max()]
ax.plot(diag, diag, "r--", linewidth=1.8, label="Perfect prediction (y = x)")
ax.set_xlabel("Actual items_sold", fontsize=12)
ax.set_ylabel("Predicted items_sold", fontsize=12)
ax.set_title("Random Forest — Parity Plot\n(Predicted vs Actual)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Random Forest Results

**RMSE = 31.68 | MAE = 24.93**

Linear Regression outperforms Random Forest on this test set — a result that is worth pausing on. On a temporal split, Random Forest can suffer from **distribution shift**: it is a memory-based model that interpolates within the training distribution, so when the test period (Jun–Dec 2024) contains date-driven patterns not well-represented in the training data, it extrapolates poorly. Linear Regression, by contrast, captures global linear trends that generalise more reliably across time. This is a well-known characteristic of tree-based models on time-series evaluation.

In [ ]:
# ── Feature Importances ────────────────────────────────────────────────────────
ohe = rf_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
ohe_feature_names = list(ohe.get_feature_names_out(cat_cols))
all_feature_names = ohe_feature_names + num_cols

importances = rf_pipeline.named_steps["model"].feature_importances_
imp_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance": importances
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Random Forest — Feature Importances (all features):")
print(imp_df.to_string(index=False))

print("\n─── Top 5 Most Influential Features ───")
top5 = imp_df.head(5)
for _, row in top5.iterrows():
    print(f"  {row['feature']:<30}  {row['importance']:.4f}")

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(imp_df["feature"][:10][::-1], imp_df["importance"][:10][::-1],
               color="steelblue", edgecolor="white")
ax.set_xlabel("Feature Importance (Mean Decrease in Impurity)", fontsize=11)
ax.set_title("Random Forest — Top 10 Feature Importances", fontsize=13, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

### Top 5 Feature Importances — Interpretation

| Rank | Feature | Importance | Business Meaning |
|------|---------|------------|-----------------|
| 1 | `store_size_small` | 0.184 | Small stores sell considerably fewer items — store size is the strongest structural predictor |
| 2 | `is_festival` | 0.174 | Festival periods drive the largest demand surge, dominating all other contextual signals |
| 3 | `location_type_urban` | 0.141 | Urban stores have systematically higher footfall and basket conversion than semi-urban/rural |
| 4 | `day_of_week` | 0.093 | Within-week patterns are significant — weekend vs weekday differences drive meaningful volume swings |
| 5 | `store_id` | 0.062 | Individual store-level effects persist after controlling for size and location, reflecting local demographics |

**Key takeaway:** The two most important predictors are **store size** and **festival timing** — structural and contextual factors that the marketing team can account for in promotion planning. `promotion_type` features do not appear in the top 5, which suggests that under the current data, store characteristics and calendar effects explain more variance in items sold than the choice of promotion alone.